# Benchmark do GIVP C++ - Funcoes Classicas

Notebook com exemplos de benchmark da implementacao header-only C++ (`cpp/include/givp`).
Cada experimento gera um programa C++ temporario, compila e executa.

## 1. Setup

In [ ]:
"""Benchmark helper utilities for compiling and running temporary C++ examples."""

from __future__ import annotations

import os
import shutil
import subprocess
from pathlib import Path

ROOT = Path.cwd().resolve().parents[1]
CPP_DIR = ROOT / "cpp"
WORK_DIR = ROOT / "Notebooks" / "_cpp_examples"
WORK_DIR.mkdir(parents=True, exist_ok=True)


def find_cpp_compiler() -> str:
    """Return the first available C++ compiler executable path."""
    for candidate in ("g++", "clang++", "cl"):
        path = shutil.which(candidate)
        if path:
            return path
    raise RuntimeError("Nenhum compilador C++ encontrado (g++/clang++/cl).")


CXX = find_cpp_compiler()
print(f"Projeto raiz: {ROOT}")
print(f"Diretorio C++: {CPP_DIR}")
print(f"Compilador: {CXX}")

## 2. Helpers

In [ ]:
def compile_and_run_cpp(name: str, source: str) -> str:
    """Compile a temporary C++ source and return the program stdout."""
    src = WORK_DIR / f"{name}.cpp"
    exe = WORK_DIR / (f"{name}.exe" if os.name == "nt" else name)
    src.write_text(source, encoding="utf-8")

    include_dir = CPP_DIR / "include"

    if Path(CXX).name.lower().startswith("cl"):
        cmd = [CXX, "/std:c++17", f"/I{include_dir}", str(src), f"/Fe:{exe}"]
    else:
        cmd = [
            CXX,
            "-std=c++17",
            "-O2",
            "-I",
            str(include_dir),
            str(src),
            "-o",
            str(exe),
        ]

    try:
        subprocess.run(cmd, capture_output=True, text=True, check=True)  # noqa: S603
    except subprocess.CalledProcessError as err:
        raise RuntimeError(err.stderr or err.stdout or str(err)) from err

    try:
        run = subprocess.run([str(exe)], capture_output=True, text=True, check=True)  # noqa: S603
    except subprocess.CalledProcessError as err:
        raise RuntimeError(err.stderr or err.stdout or str(err)) from err

    return run.stdout.strip()

## 3. Sphere e Rosenbrock

In [ ]:
SPHERE_ROSEN = r"""
#include <cmath>
#include <iostream>
#include <vector>
#include <givp/api.hpp>
#include <givp/config.hpp>

double sphere(const std::vector<double>& x) {
  double s = 0.0;
  for (double v : x) s += v * v;
  return s;
}

double rosenbrock(const std::vector<double>& x) {
  double s = 0.0;
  for (std::size_t i = 0; i + 1 < x.size(); ++i) {
    const double a = x[i + 1] - x[i] * x[i];
    const double b = 1.0 - x[i];
    s += 100.0 * a * a + b * b;
  }
  return s;
}

int main() {
  givp::GivpConfig cfg;
  cfg.max_iterations = 60;
  cfg.alpha = 0.15;
  cfg.vnd_iterations = 30;
  cfg.ils_iterations = 8;
  cfg.early_stop_threshold = 40;
  cfg.seed = 42;

  auto b1 = std::vector<std::pair<double, double>>(10, {-5.12, 5.12});
  auto r1 = givp::givp(sphere, b1, cfg);

  auto b2 = std::vector<std::pair<double, double>>(5, {-2.048, 2.048});
  auto r2 = givp::givp(rosenbrock, b2, cfg);

  std::cout << "Sphere best_fun=" << r1.fun << " nfev=" << r1.nfev << "\n";
  std::cout << "Rosenbrock best_fun=" << r2.fun << " nfev=" << r2.nfev << "\n";
}
"""

print(compile_and_run_cpp("benchmark_sphere_rosen", SPHERE_ROSEN))

## 4. Rastrigin e Ackley

In [ ]:
RAST_ACK = r"""
#include <cmath>
#include <iostream>
#include <vector>
#include <givp/api.hpp>
#include <givp/config.hpp>

double rastrigin(const std::vector<double>& x) {
  const double pi = 3.14159265358979323846;
  double s = 10.0 * static_cast<double>(x.size());
  for (double v : x) s += v * v - 10.0 * std::cos(2.0 * pi * v);
  return s;
}

double ackley(const std::vector<double>& x) {
  const double pi = 3.14159265358979323846;
  const double n = static_cast<double>(x.size());
  double s1 = 0.0;
  double s2 = 0.0;
  for (double v : x) {
    s1 += v * v;
    s2 += std::cos(2.0 * pi * v);
  }
  s1 /= n;
  s2 /= n;
  return -20.0 * std::exp(-0.2 * std::sqrt(s1)) - std::exp(s2) + 20.0 + std::exp(1.0);
}

int main() {
  givp::GivpConfig cfg;
  cfg.seed = 42;
  cfg.max_iterations = 80;
  cfg.alpha = 0.12;
  cfg.vnd_iterations = 50;
  cfg.ils_iterations = 10;
  cfg.early_stop_threshold = 60;

  auto b_rast = std::vector<std::pair<double, double>>(10, {-5.12, 5.12});
  auto b_ack = std::vector<std::pair<double, double>>(10, {-32.768, 32.768});

  auto r_rast = givp::givp(rastrigin, b_rast, cfg);
  auto r_ack = givp::givp(ackley, b_ack, cfg);

  std::cout << "Rastrigin best_fun=" << r_rast.fun << " nfev=" << r_rast.nfev << "\n";
  std::cout << "Ackley best_fun=" << r_ack.fun << " nfev=" << r_ack.nfev << "\n";
}
"""

print(compile_and_run_cpp("benchmark_rastrigin_ackley", RAST_ACK))

## 5. Consolidacao simples

In [ ]:
print(
    "Para analise estatistica completa, use o notebook de comparacao com literatura (C++)."
)